In [ ]:
#!/usr/bin/env python3
"""
MOFA2 + Cox survival TEST pipeline for TCGA-BRCA.

Run train_mofa_brca_survival.R first. This notebook/script then:
  - loads the frozen BRCA survival split
  - loads full BRCA matrices in frozen sample order
  - loads MOFA2 HDF5 weights from training
  - projects TEST samples into MOFA factors via joint ridge projection
  - applies saved Cox coefficients from training
  - saves AMP-compatible survival outputs under ../results/brca_survival/mofa_test
"""

from __future__ import annotations

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd


In [ ]:
# -----------------------------
# Config
# -----------------------------

matrices_dir = Path("../matrices")
data_dir     = Path("../data")
splits_dir   = Path("../splits")
model_dir    = Path("../models/mofa_survival")
results_dir  = Path("../results/brca_survival/mofa_test")
results_dir.mkdir(parents=True, exist_ok=True)

SPLIT_TAG = "brca_survival"
RIDGE_LAM = 1e-3

mofa_hdf5 = model_dir / f"{SPLIT_TAG}_mofa_survival_model.hdf5"
coef_path = model_dir / f"{SPLIT_TAG}_mofa_cox_coefficients.csv"

print("Results dir:", results_dir.resolve())


In [ ]:
# -----------------------------
# Helpers
# -----------------------------

def load_index_csv_0based(path: Path, n_total: int) -> np.ndarray:
    df = pd.read_csv(path)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) == 0:
        raise ValueError(f"No numeric index column in {path}")
    idx = df[num_cols[0]].to_numpy()
    if np.isnan(idx).any():
        raise ValueError(f"NaN index in {path}")
    idx_int = idx.astype(np.int64)
    if not np.allclose(idx, idx_int):
        raise ValueError(f"Non-integer index in {path}")

    mn, mx = int(idx_int.min()), int(idx_int.max())
    if mn == 0 and mx == n_total - 1:
        idx0 = idx_int
    elif mn == 1 and mx == n_total:
        idx0 = idx_int - 1
    else:
        idx0 = idx_int

    if (idx0 < 0).any() or (idx0 >= n_total).any():
        raise ValueError(f"Out-of-range index in {path}: [{idx0.min()}, {idx0.max()}], n={n_total}")
    return idx0


def load_mofa_weights_from_hdf5(path: Path) -> dict[str, np.ndarray]:
    import h5py

    with h5py.File(path, "r") as f:
        def iter_groups(g, prefix=""):
            for key, obj in g.items():
                p = f"{prefix}/{key}"
                if isinstance(obj, h5py.Group):
                    yield p, obj
                    yield from iter_groups(obj, p)

        candidates = []
        for p, g in iter_groups(f):
            lp = p.lower()
            if lp.endswith("/expectations/w") or ("/expectations/" in lp and lp.endswith("/w")):
                candidates.append((p, g))
        if not candidates:
            raise RuntimeError(f"Could not find expectations/W in {path}")

        candidates.sort(key=lambda x: len(x[0]))
        _, w_group = candidates[0]
        out = {}
        for view_name, obj in w_group.items():
            if hasattr(obj, "shape"):
                W = np.array(obj, dtype=np.float32)
            else:
                found = None
                for key in ("value", "mean", "data", "W"):
                    if key in obj:
                        found = obj[key]
                        break
                if found is None:
                    continue
                W = np.array(found, dtype=np.float32)
            if W.shape[0] < W.shape[1]:
                W = W.T
            out[view_name] = W
        if not out:
            raise RuntimeError(f"Found W group in {path}, but extracted no views")
        return out


def orient_mofa_weights(
    X_by_view: dict[str, np.ndarray],
    W_by_view: dict[str, np.ndarray],
) -> dict[str, np.ndarray]:
    """Orient each MOFA W matrix to features x factors using the observed feature count."""
    out = {}
    for v, X in X_by_view.items():
        W = np.asarray(W_by_view[v], dtype=np.float32)
        p = X.shape[0]
        if W.shape[0] == p:
            out[v] = W
        elif W.shape[1] == p:
            out[v] = W.T
        else:
            raise ValueError(f"[{v}] feature mismatch: X={X.shape}, W={W.shape}; neither W dimension equals {p}")
    return out


def joint_ridge_project(X_by_view: dict[str, np.ndarray], W_by_view: dict[str, np.ndarray], lam: float) -> np.ndarray:
    views = list(X_by_view.keys())
    K = W_by_view[views[0]].shape[1]
    n = X_by_view[views[0]].shape[1]
    A = lam * np.eye(K, dtype=np.float64)
    B = np.zeros((n, K), dtype=np.float64)

    for v in views:
        X = X_by_view[v]
        W = W_by_view[v]
        if X.shape[0] != W.shape[0]:
            raise ValueError(f"[{v}] feature mismatch: X={X.shape}, W={W.shape}")
        B += X.T @ W
        A += W.T @ W
    return (B @ np.linalg.solve(A, np.eye(K))).astype(np.float32)


def concordance_index_survival(time: np.ndarray, event: np.ndarray, log_risk: np.ndarray) -> float:
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    score = -np.asarray(log_risk, dtype=float)  # higher score means longer survival, matching lifelines usage
    concordant = 0.0
    permissible = 0.0
    n = len(time)
    for i in range(n):
        for j in range(i + 1, n):
            if time[i] == time[j]:
                continue
            if time[i] < time[j] and event[i] == 1:
                permissible += 1.0
                if score[i] < score[j]:
                    concordant += 1.0
                elif score[i] == score[j]:
                    concordant += 0.5
            elif time[j] < time[i] and event[j] == 1:
                permissible += 1.0
                if score[j] < score[i]:
                    concordant += 1.0
                elif score[j] == score[i]:
                    concordant += 0.5
    return float(concordant / permissible) if permissible else float("nan")


In [ ]:
# -----------------------------
# Load frozen split and data
# -----------------------------

for p in [mofa_hdf5, coef_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Run train_mofa_brca_survival.R first.")

sample_ids = pd.read_csv(splits_dir / f"{SPLIT_TAG}_sample_ids.csv")["sample_id"].tolist()
idx_te = load_index_csv_0based(splits_dir / f"{SPLIT_TAG}_test_idx.csv", len(sample_ids))
test_ids = [sample_ids[i] for i in idx_te]

rna_df  = pd.read_csv(matrices_dir / "RNA_X_full.csv", index_col=0)
meth_df = pd.read_csv(matrices_dir / "Methylation_X_full.csv", index_col=0)
cnv_df  = pd.read_csv(matrices_dir / "CNV_X_ld_features.csv", index_col=0)

surv_raw = pd.read_csv(data_dir / "BRCA_survival.tsv", sep="\t", index_col=0)
surv_df = surv_raw[["OS", "OS.time"]].copy()
surv_df.columns = ["event", "time"]
surv_df["event"] = pd.to_numeric(surv_df["event"], errors="coerce")
surv_df["time"] = pd.to_numeric(surv_df["time"], errors="coerce")
surv_df = surv_df.dropna(subset=["event", "time"])
surv_df = surv_df[surv_df["time"] > 0]

X_by_view = {
    "RNA": rna_df.loc[test_ids].to_numpy(dtype=np.float32).T,
    "Methylation": meth_df.loc[test_ids].to_numpy(dtype=np.float32).T,
    "CNV": cnv_df.loc[test_ids].to_numpy(dtype=np.float32).T,
}
event_te = surv_df.loc[test_ids, "event"].to_numpy(dtype=np.float32)
time_te = surv_df.loc[test_ids, "time"].to_numpy(dtype=np.float32)

print(f"Loaded TEST samples: {len(test_ids)} | events: {int(event_te.sum())}")


In [ ]:
# -----------------------------
# Project MOFA factors and apply Cox coefficients
# -----------------------------

W_by_view = load_mofa_weights_from_hdf5(mofa_hdf5)
for v in X_by_view:
    if v not in W_by_view:
        raise KeyError(f"View {v!r} missing from MOFA weights. Found: {list(W_by_view)}")
W_by_view = orient_mofa_weights(X_by_view, W_by_view)
print("MOFA W shapes after orientation:", {v: W.shape for v, W in W_by_view.items()})

Z_test = joint_ridge_project(X_by_view, W_by_view, lam=RIDGE_LAM)

coef_df = pd.read_csv(coef_path)
coef_df = coef_df.sort_values("factor", key=lambda s: s.str.extract(r"(\d+)")[0].astype(int))
beta = coef_df["coef"].to_numpy(dtype=np.float64)
if Z_test.shape[1] != len(beta):
    raise ValueError(f"Factor/coef mismatch: Z_test has {Z_test.shape[1]} factors, beta has {len(beta)}")

log_risk_test = Z_test @ beta
c_test = concordance_index_survival(time_te, event_te, log_risk_test)
print(f"MOFA2 + CoxPH TEST C-index: {c_test:.4f}")


In [ ]:
# -----------------------------
# Save outputs
# -----------------------------

np.save(results_dir / "Z_test.npy", Z_test)
np.save(results_dir / "log_risk_test.npy", log_risk_test.astype(np.float32))
np.save(results_dir / "event_test.npy", event_te)
np.save(results_dir / "time_test.npy", time_te)
np.save(results_dir / "idx_test.npy", idx_te.astype(int))

pred_df = pd.DataFrame({
    "sample_id": test_ids,
    "time": time_te,
    "event": event_te,
    "log_risk": log_risk_test,
    "risk_group": np.where(log_risk_test >= np.median(log_risk_test), "High risk", "Low risk"),
})
pred_df.to_csv(results_dir / "test_predictions.csv", index=False)

report = {
    "c_index_test": float(c_test),
    "n_test": int(len(test_ids)),
    "n_events_test": int(event_te.sum()),
    "run": {
        "method": "MOFA2 + CoxPH",
        "task": "survival",
        "split_tag": SPLIT_TAG,
        "ridge_projection_lambda": float(RIDGE_LAM),
        "mofa_hdf5": str(mofa_hdf5),
        "cox_coefficients": str(coef_path),
        "views": list(X_by_view.keys()),
    },
    "timestamp": datetime.now().isoformat(timespec="seconds"),
}
with open(results_dir / "metrics.json", "w") as f:
    json.dump(report, f, indent=2)

print("Saved to:", results_dir.resolve())
print(json.dumps(report, indent=2))
